# Feature engineering for business survival analysis

This notebook transforms raw Calgary Business Licence data into features suitable for
survival modelling and binary classification. We create survival duration and event
columns, encode business type, extract location-level aggregates, and handle
censored observations.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

sys.path.insert(0, '..')
from src.data_loader import (
    fetch_business_licences,
    _parse_dates,
    _create_survived_column,
    _extract_business_category,
    _encode_home_occupation,
    _build_community_features,
    ACTIVE_STATUSES,
    INACTIVE_STATUSES,
)

pd.set_option('display.max_columns', 40)
print('Setup complete.')

## 1. Load raw business licence data

We pull the raw data from the local cache (or the Socrata API) before applying
any transformations, so we can inspect each step individually.

In [ ]:
raw = fetch_business_licences(limit=50000)
print(f'Raw records: {len(raw):,}')
print(f'Columns: {list(raw.columns)}')
raw.head(3)

## 2. Create survival duration and event columns

Survival analysis requires two key fields:

- **Duration** (`business_age_days`): the number of days between the first issue date and either the expiry date or today.
- **Event observed** (`event_observed`): 1 if the business closed (the event happened), 0 if the business is still active (right-censored).

We derive both using `_parse_dates` and `_create_survived_column` from the data loader.

In [ ]:
df = _parse_dates(raw)
print('Date columns added:', [c for c in df.columns if c not in raw.columns])
print(f'business_age_days range: {df["business_age_days"].min():.0f} -- {df["business_age_days"].max():.0f} days')
df[['first_iss_dt', 'exp_dt', 'business_age_days', 'issue_year', 'issue_month']].describe()

In [ ]:
df = _create_survived_column(df)

# event_observed is the complement of survived: 1 = closed, 0 = censored
df['event_observed'] = 1 - df['survived']

print('Survival target distribution:')
print(df['survived'].value_counts())
print(f'\nCensored (still active): {(df["event_observed"] == 0).sum():,}')
print(f'Event observed (closed): {(df["event_observed"] == 1).sum():,}')

## 3. Handle censored observations

Right-censoring occurs when a business is still active at the time of data
extraction. For these cases the duration is the time from first issue to today,
and `event_observed` is set to 0. The Kaplan-Meier and Cox models in
`src/model.py` account for this automatically.

In [ ]:
censored = df[df['event_observed'] == 0]
observed = df[df['event_observed'] == 1]

print(f'Censored observations: {len(censored):,} ({len(censored)/len(df)*100:.1f}%)')
print(f'Observed closures:     {len(observed):,} ({len(observed)/len(df)*100:.1f}%)')
print(f'\nMedian duration (censored):  {censored["business_age_days"].median():.0f} days')
print(f'Median duration (observed):  {observed["business_age_days"].median():.0f} days')

In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=censored['business_age_days'] / 365, name='Censored (active)',
                           opacity=0.7, marker_color='steelblue'))
fig.add_trace(go.Histogram(x=observed['business_age_days'] / 365, name='Observed (closed)',
                           opacity=0.7, marker_color='salmon'))
fig.update_layout(title='Duration distribution by censoring status',
                  xaxis_title='Business age (years)', yaxis_title='Count',
                  barmode='overlay', height=400)
fig.show()

## 4. Encode business type

The `licencetypes` field contains a dash-separated string. We extract the leading
token as a simplified `business_category` and then label-encode it for use in
tree-based classifiers.

In [ ]:
df = _extract_business_category(df)

print(f'Unique business categories: {df["business_category"].nunique()}')
df['business_category'].value_counts().head(15)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['business_category_enc'] = le.fit_transform(df['business_category'].astype(str))

print(f'Encoded range: {df["business_category_enc"].min()} -- {df["business_category_enc"].max()}')
df[['business_category', 'business_category_enc']].drop_duplicates().head(10)

## 5. Extract location features

Community-level aggregates capture the local business environment. The data loader
computes `business_count`, `business_diversity`, and `avg_business_age` per community
district via `_build_community_features`.

In [ ]:
df = _encode_home_occupation(df)
df = _build_community_features(df)

location_cols = ['comdistnm', 'business_count', 'business_diversity', 'avg_business_age', 'is_home_occupation']
df[location_cols].drop_duplicates(subset=['comdistnm']).sort_values('business_count', ascending=False).head(15)

In [ ]:
fig = px.scatter(
    df.drop_duplicates(subset=['comdistnm']),
    x='business_diversity',
    y='business_count',
    size='avg_business_age',
    hover_name='comdistnm',
    title='Community business count vs. diversity',
    labels={'business_diversity': 'Distinct categories', 'business_count': 'Total businesses'},
)
fig.update_layout(height=450)
fig.show()

## 6. Feature distributions

Before modelling we inspect the distribution of each numeric feature to identify
skewness, outliers, or need for transformations.

In [ ]:
feature_cols = [
    'business_age_days', 'is_home_occupation', 'business_count',
    'business_diversity', 'avg_business_age', 'issue_year',
    'issue_month', 'business_category_enc',
]
df[feature_cols].describe().T

In [ ]:
from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=4, subplot_titles=feature_cols)
for i, col in enumerate(feature_cols):
    row, col_idx = i // 4 + 1, i % 4 + 1
    fig.add_trace(
        go.Histogram(x=df[col], name=col, marker_color='steelblue', showlegend=False),
        row=row, col=col_idx,
    )
fig.update_layout(title='Feature distributions', height=500)
fig.show()

## 7. Final engineered dataset

The processed dataframe now contains everything needed for survival analysis
(duration + event columns) and binary classification (feature matrix + survived target).

In [ ]:
final_cols = feature_cols + ['survived', 'event_observed', 'business_age_days', 'comdistnm', 'business_category']
final_cols = list(dict.fromkeys(final_cols))  # deduplicate while preserving order

print(f'Engineered dataset: {df.shape[0]:,} rows x {len(final_cols)} selected columns')
print(f'Target balance: {df["survived"].mean():.2%} survived')
df[final_cols].head(10)

## Summary

Feature engineering steps completed:

1. Parsed issue and expiry dates to compute `business_age_days` as the survival duration.
2. Created `survived` (binary target) and `event_observed` (censoring indicator) from licence status.
3. Extracted a simplified `business_category` and label-encoded it.
4. Encoded home-occupation indicator as a binary feature.
5. Built community-level aggregate features: business count, diversity, and average age.
6. Verified feature distributions and checked for skewness.

The resulting dataset feeds directly into the Kaplan-Meier, Cox PH, Random Forest,
and XGBoost models in `03_modeling.ipynb`.